## 🎯 Working Example (Run after starting TWS/Gateway)

In [1]:
"""
Complete working example of TWS API low-level handshake.
Prerequisites:
1. TWS or IB Gateway must be running
2. API must be enabled in Global Configuration
3. Socket port = 7497 (or update below)
4. 127.0.0.1 must be in Trusted IP Addresses
"""

import socket
import struct
import logging
import threading
from ibapi import comm

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Configuration
HOST = '127.0.0.1'
PORT = 7497
MIN_CLIENT_VER = 100
MAX_CLIENT_VER = 203

def tws_handshake():
    """Perform TWS API handshake and return server version and connection time."""
    
    # Create socket with timeout
    soc = socket.socket()
    soc.settimeout(2)
    
    try:
        # Connect to TWS
        logger.info(f"Connecting to {HOST}:{PORT}...")
        soc.connect((HOST, PORT))
        logger.info(f"Connected: {soc.getsockname()} -> {soc.getpeername()}")
        
        # Build handshake message
        v100prefix = "API\0"
        v100version = f"v{MIN_CLIENT_VER}..{MAX_CLIENT_VER}"
        msg = comm.make_initial_msg(v100version)
        msg_with_prefix = str.encode(v100prefix, "ascii") + msg
        
        # Send handshake
        soc.send(msg_with_prefix)
        logger.info(f"Sent handshake: {msg_with_prefix}")
        
        # Receive server response
        buf = b""
        fields = []
        attempts = 0
        max_attempts = 5
        
        while len(fields) < 2 and attempts < max_attempts:
            attempts += 1
            try:
                chunk = soc.recv(4096)
                if not chunk:
                    raise ConnectionError("Server closed connection")
                
                buf += chunk
                logger.debug(f"Received {len(chunk)} bytes")
                
                # Parse message
                if len(buf) >= 4:
                    (size, msg, rest) = comm.read_msg(buf)
                    if size > 0 and msg:
                        fields = comm.read_fields(msg)
                        if len(fields) >= 2:
                            break
                        buf = rest
                        
            except socket.timeout:
                logger.debug(f"Timeout attempt {attempts}")
                continue
        
        if len(fields) >= 2:
            server_version = fields[0].decode() if isinstance(fields[0], bytes) else str(fields[0])
            conn_time = fields[1].decode() if isinstance(fields[1], bytes) else str(fields[1])
            
            print("\n" + "="*50)
            print("✅ HANDSHAKE SUCCESSFUL!")
            print("="*50)
            print(f"Server Version: {server_version}")
            print(f"Connection Time: {conn_time}")
            print("="*50 + "\n")
            
            return (server_version, conn_time)
        else:
            raise TimeoutError("No response from server - is TWS/Gateway running?")
            
    except ConnectionRefusedError:
        logger.error("❌ Connection refused - TWS/Gateway not running on port " + str(PORT))
        raise
    except socket.timeout:
        logger.error("❌ Timeout - TWS/Gateway not responding (is API enabled?)")
        raise
    finally:
        soc.close()
        logger.debug("Socket closed")

# Run the handshake
try:
    result = tws_handshake()
except Exception as e:
    print(f"\n❌ FAILED: {e}")


INFO: Connecting to 127.0.0.1:7497...
INFO: Connected: ('127.0.0.1', 47988) -> ('127.0.0.1', 7497)
INFO: Sent handshake: b'API\x00\x00\x00\x00\tv100..203'



✅ HANDSHAKE SUCCESSFUL!
Server Version: 203
Connection Time: 20251124 17:59:51 Central European Standard Time



In [2]:
from ibapi.client import *
from ibapi.wrapper import *
import time
import threading

class TestApp(EClient, EWrapper):
  def __init__(self):
    EClient.__init__(self, self)
  
  def nextValidId(self, orderId):
    self.orderId = orderId
  
  def nextId(self):
    self.orderId += 1
    return self.orderId

  def error(self, reqId, errorCode, errorString, advancedOrderReject, *args):
    print(f"reqId: {reqId}, errorCode: {errorCode}, errorString: {errorString}, orderReject: {advancedOrderReject}")

  def contractDetails(self, reqId, contractDetails):
    attrs = vars(contractDetails)
    print("\n".join(f"{name}: {value}" for name,value in attrs.items()))
    # print(contractDetails.contract)

  def contractDetailsEnd(self, reqId):
    print("End of contract details")
    self.disconnect()

app = TestApp()
app.connect("127.0.0.1", 7497, 0)
thr = threading.Thread(target=app.run)
thr.start()
time.sleep(1)

mycontract = Contract()
# Stock
# mycontract.symbol = "AAPL"
# mycontract.secType = "STK"
# mycontract.currency = "USD"
# mycontract.exchange = "SMART"
# mycontract.primaryExchange = "NASDAQ"

# Future
# mycontract.symbol = "ES"
# mycontract.secType = "FUT"
# mycontract.currency = "USD"
# mycontract.exchange = "CME"
# mycontract.lastTradeDateOrContractMonth = 202412

# Option
mycontract.symbol = "AAPL"
mycontract.secType = "STK"
mycontract.currency = "USD"
mycontract.exchange = "SMART"
# mycontract.lastTradeDateOrContractMonth = 202412
# mycontract.right = "P"
# mycontract.tradingClass = "SPXW"
# mycontract.strike = 5300

app.reqContractDetails(app.nextId(), mycontract)
time.sleep(3)
app.disconnect()

INFO: sent startApi
INFO: REQUEST startApi {}
INFO: SENDING startApi b'\x00\x00\x00\t\x00\x00\x00G2\x000\x00\x00'
INFO: ANSWER connectAck {}
INFO: ANSWER managedAccounts {'accountsList': 'DU6968828'}
INFO: ANSWER errorProtoBuf {'errorMessageProto': id: -1
errorTime: 1764003591269
errorCode: 2104
errorMsg: "Market data farm connection is OK:usfarm"
}
INFO: ANSWER errorProtoBuf {'errorMessageProto': id: -1
errorTime: 1764003591270
errorCode: 2107
errorMsg: "HMDS data farm connection is inactive but should be available upon demand.ushmds"
}
INFO: ANSWER errorProtoBuf {'errorMessageProto': id: -1
errorTime: 1764003591270
errorCode: 2158
errorMsg: "Sec-def data farm connection is OK:secdefil"
}


reqId: -1, errorCode: 1764003591269, errorString: 2104, orderReject: Market data farm connection is OK:usfarm
reqId: -1, errorCode: 1764003591270, errorString: 2107, orderReject: HMDS data farm connection is inactive but should be available upon demand.ushmds
reqId: -1, errorCode: 1764003591270, errorString: 2158, orderReject: Sec-def data farm connection is OK:secdefil


INFO: REQUEST reqContractDetails {'reqId': 2, 'contract': 131350536713808: ConId: 0, Symbol: AAPL, SecType: STK, LastTradeDateOrContractMonth: , Strike: , Right: , Multiplier: , Exchange: SMART, PrimaryExchange: , Currency: USD, LocalSymbol: , TradingClass: , IncludeExpired: False, SecIdType: , SecId: , Description: , IssuerId: Combo:}
INFO: SENDING reqContractDetails b'\x00\x00\x00)\x00\x00\x00\t8\x002\x000\x00AAPL\x00STK\x00\x00\x00\x00\x00SMART\x00\x00USD\x00\x00\x000\x00\x00\x00\x00'
INFO: disconnecting
INFO: ANSWER connectionClosed {}


contract: ConId: 265598, Symbol: AAPL, SecType: STK, LastTradeDateOrContractMonth: , Strike: 0, Right: , Multiplier: , Exchange: SMART, PrimaryExchange: NASDAQ, Currency: USD, LocalSymbol: AAPL, TradingClass: NMS, IncludeExpired: False, SecIdType: , SecId: , Description: , IssuerId: Combo:
marketName: NMS
minTick: 0.01
orderTypes: ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AON,AVGCOST,BASKET,BENCHPX,CASHQTY,COND,CONDORDER,DARKONLY,DARKPOLL,DAY,DEACT,DEACTDIS,DEACTEOD,DIS,DUR,GAT,GTC,GTD,GTT,HID,IBKRATS,ICE,IMB,IOC,LIT,LMT,LOC,MIDPX,MIT,MKT,MOC,MTL,NGCOMB,NODARK,NONALGO,OCA,OPG,OPGREROUT,PEGBENCH,PEGMID,POSTATS,POSTONLY,PREOPGRTH,PRICECHK,REL,REL2MID,RELPCTOFS,RPI,RTH,SCALE,SCALEODD,SCALERST,SIZECHK,SMARTSTG,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,SWEEP,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF
validExchanges: SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,NASDAQ,DRCTEDGE,BEX,BATS,EDGEA,BYX,IEX,EDGX,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X
priceMagnifier: 1
underConId: 0
longN